# International Tourism Receipts
## Processing data to find out tourism information
Dataset source: ***https://www.kaggle.com/datasets/abdulhamitcelik/international-tourism-receipts***


### Setup & Imports

In [5]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Set visualisation style
sns.set_theme(style='darkgrid')
plt.style.use('seaborn-v0_8-colorblind')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


### Load dataset

In [15]:
# Load dataset
print("Loading tourism dataset...")
df = pd.read_csv('tourism-recipts.csv')

print("✅ TOURISM RECEIPTS DATASET")
print("="*60)


# Explore dataset information
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Years covered: {df['year'].min()} - {df['year'].max()}")
print(f"Number of countries: {df['name'].nunique()}")
print(f"Time range: {df['year'].nunique()} years")
print(f"Value range: ${df['value_$'].min():,.0f} - ${df['value_$'].max():,.0f}")


print("\n📊 First 5 rows:")
print(df.head())
print("\n📊 Data types:")
print(df.dtypes)
print("\n📊 Last 5 rows:")
print(df.tail())

Loading tourism dataset...
✅ TOURISM RECEIPTS DATASET
Shape: 4,156 rows, 4 columns
Years covered: 1995 - 2020
Number of countries: 203
Time range: 26 years
Value range: $100,000 - $241,984,000,000

📊 First 5 rows:
          name code  year      value_$
0  Afghanistan  AFG  2008  57000000.00
1  Afghanistan  AFG  2009  89000000.00
2  Afghanistan  AFG  2010 147000000.00
3  Afghanistan  AFG  2011 165000000.00
4  Afghanistan  AFG  2012 167000000.00

📊 Data types:
name           str
code           str
year         int64
value_$    float64
dtype: object

📊 Last 5 rows:
          name code  year      value_$
4151  Zimbabwe  ZWE  2016 194000000.00
4152  Zimbabwe  ZWE  2017 158000000.00
4153  Zimbabwe  ZWE  2018 191000000.00
4154  Zimbabwe  ZWE  2019 285000000.00
4155  Zimbabwe  ZWE  2020  66000000.00


### Initial Data Assessment

In [27]:
# Understanding the dataset
print("🔍 DATA QUALITY CHECK")
print("="*60)

# Check for missing values
print("\n1️⃣ MISSING VALUES:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("    ✅ No missing values found!")
else:
    print(missing[missing > 0])

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"\n2️⃣ DUPLICATES: {duplicates}")

# Check for negative values
negative_values = (df['value_$'] < 0).sum()
print(f"\n3️⃣ NEGATIVE VALUES: {negative_values}")

# Check value distribution
print("\n4️⃣ VALUE STATISTICS:")
print(f"    Mean: ${df['value_$'].mean():,.0f}")
print(f"    Median: ${df['value_$'].median():,.0f}")
print(f"    Std Dev: ${df['value_$'].std():,.0f}")

# Check years per country
years_per_country = df.groupby('name')['year'].count()
print("\n5️⃣ YEARS PER COUNTRY:")
print(f"    Min: {years_per_country.min()}")
print(f"    Max: {years_per_country.max()}")
print(f"    Mean: {years_per_country.mean():.1f}")

# List countries with incomplete data
incomplete = years_per_country[years_per_country < df['year'].nunique()].index.tolist()
if incomplete:
    print(f"\n⚠️ Countries with missing years: {len(incomplete)}")
    print(f"    First 5: {incomplete[:5]}")

🔍 DATA QUALITY CHECK

1️⃣ MISSING VALUES:
    ✅ No missing values found!

2️⃣ DUPLICATES: 0

3️⃣ NEGATIVE VALUES: 0

4️⃣ VALUE STATISTICS:
    Mean: $4,919,326,950
    Median: $749,500,000
    Std Dev: $14,980,536,309

5️⃣ YEARS PER COUNTRY:
    Min: 1
    Max: 26
    Mean: 20.5

⚠️ Countries with missing years: 126
    First 5: ['Afghanistan', 'Algeria', 'American Samoa', 'Andorra', 'Antigua and Barbuda']


### Create Additional Features

In [32]:
# Feature engineering
print("🔧 FEATURE ENGINEERNG")
print("="*60)

# Create a copy for analysis
df_clean = df.copy()

# Convert values to millions for easier reading
df_clean['value_millions'] = df_clean['value_$'] / 1_000_000

# Calculate year-over-year growth
df_clean = df_clean.sort_values(['name', 'year'])
df_clean['prev_year_value'] = df_clean.groupby('name')['value_$'].shift(1)
df_clean['growth_rate'] = ((df_clean['value_$'] - df_clean['prev_year_value']) / df_clean['prev_year_value'] * 100)
df_clean['growth_rate'] = df_clean['growth_rate'].fillna(0)   # First year has no growth

# Add a decade column
df_clean['decade'] = (df_clean['year'] // 10) * 10

# Flag for growth (positive/negative)
df_clean['is_growing'] = df_clean['growth_rate'] > 0

# Categorise growth
def categorise_growth(rate):
    if rate > 20:
        return 'High Growth (>20%)'
    elif rate > 0:
        return 'Postive Growth (0 - 20%)'
    elif rate > -20:
        return 'Mild Decline (-20% to 0)'
    else:
        return 'Sharp Decline (< -20%)'

df_clean['growth_category'] = df_clean['growth_rate'].apply(categorise_growth)

print("✅ Added columns:")
print("    - value_millions (values in millions)")
print("    - prev_year_value (previous year's receipts)")
print("    - growth_rate (year-over-year % change)")
print("    - decade (grouped by decade)")
print("    - is_growing (boolean flag)")
print("    - growth_category (categorized growth)")  

print(f"\n📊 New shape: {df_clean.shape}")
df_clean.head()

🔧 FEATURE ENGINEERNG
✅ Added columns:
    - value_millions (values in millions)
    - prev_year_value (previous year's receipts)
    - growth_rate (year-over-year % change)
    - decade (grouped by decade)
    - is_growing (boolean flag)
    - growth_category (categorized growth)

📊 New shape: (4156, 10)


,name,code,year,value_$,value_millions,prev_year_value,growth_rate,decade,is_growing,growth_category
0,Afghanistan,AFG,2008,57000000.00,57.00,NaN,0.00,2000,False,Mild Decline (-20% to 0)
1,Afghanistan,AFG,2009,89000000.00,89.00,57000000.00,56.14,2000,True,High Growth (>20%)
2,Afghanistan,AFG,2010,147000000.00,147.00,89000000.00,65.17,2010,True,High Growth (>20%)
3,Afghanistan,AFG,2011,165000000.00,165.00,147000000.00,12.24,2010,True,Postive Growth (0 - 20%)
4,Afghanistan,AFG,2012,167000000.00,167.00,165000000.00,1.21,2010,True,Postive Growth (0 - 20%)


### Analysing Global Trends

In [35]:
# Global trends
print("🌍 GLOBAL TOURISM RECEIPTS ANALYSIS")
print("="*60)

# Gloabl totals by year
global_totals = df_clean.groupby('year')['value_millions'].sum().reset_index()

print("\n1️⃣ GLOBAL TOURISM RECEIPTS OVER TIME:")
for _, row in global_totals.iterrows():
    year = int(row['year'])
    value = row['value_millions']
    # Creating a simple bar chart in text
    bar = '█' * int(value / 20000)  # Scale for display
    print(f"   {year}: {bar} ${value:,.0f}M")

# Growth years
print("\n2️⃣ BEST YEARS FOR GLOBAL TOURISM:")
top_years = global_totals.nlargest(5, 'value_millions')
for _, row in top_years.iterrows():
    print(f"    {int(row['year'])}: ${row['value_millions']:,.0f}M")

print("\n3️⃣ COVID-19 IMPACT (2019 vs 2020:")
if 2019 in df_clean['year'].values and 2020 in df_clean['year'].values:
    receipt_2019 = global_totals[global_totals['year'] == 2019]['value_millions'].values[0]
    receipt_2020 = global_totals[global_totals['year'] == 2020]['value_millions'].values[0]
    decline = ((receipt_2020 - receipt_2019) / receipt_2019) * 100
    print(f"    2019: ${receipt_2019:,.0f}M")
    print(f"    2020: ${receipt_2020:,.0f}M")
    print(f"    Change: {decline:.1f}%")

🌍 GLOBAL TOURISM RECEIPTS ANALYSIS

1️⃣ GLOBAL TOURISM RECEIPTS OVER TIME:
   1995: █████████████████████ $423,343M
   1996: ██████████████████████ $450,460M
   1997: █████████████████████ $435,308M
   1998: █████████████████████ $439,252M
   1999: ██████████████████████ $447,574M
   2000: ███████████████████████ $461,370M
   2001: ███████████████████████ $460,506M
   2002: ████████████████████████ $485,579M
   2003: ███████████████████████████ $540,778M
   2004: ███████████████████████████████ $637,947M
   2005: █████████████████████████████████ $661,027M
   2006: ██████████████████████████████████ $694,518M
   2007: ████████████████████████████████████████ $814,430M
   2008: █████████████████████████████████████████ $836,716M
   2009: ██████████████████████████████████████ $761,184M
   2010: ██████████████████████████████████████████ $842,844M
   2011: ██████████████████████████████████████████████ $938,397M
   2012: █████████████████████████████████████████████████ $997,591M
   2013

### Analyse Country Earnings

In [39]:
# Which countries earn the most from tourism
print("🏆 TOP TOURISM EARNERS")
print("="*50)

# Calculate recepts by country
country_totals = df_clean.groupby('name')['value_millions'].sum().sort_values(ascending=False)

print("\n1️⃣ ALL-TIME TOP 10 TOURISM EARNERS:")
for i, (country, value) in enumerate(country_totals.head(10).items(), 1):
    print(f"    {i:2d}. {country:<30} ${value:,.0f}M")

# Last year (2020) performance
latest_year = df_clean['year'].max()
latest_receipts = df_clean[df_clean['year'] == latest_year].set_index('name')['value_millions'].sort_values(ascending=False)

print("\n2️⃣ TOP 10 IN {latest_year} (COVID year):")
for i, (country, value) in enumerate(latest_receipts.head(10).items(), 1):
    print(f"    {i:2d}. {country:<30} ${value:,.0f}M")

# Fastest growing countries
avg_growth = df_clean[df_clean['growth_rate'] != 0].groupby('name')['growth_rate'].mean().sort_values(ascending=False)

print("\n3️⃣ FASTEST GROWING COUNTRIES (average annual %):")
for i, (country, growth) in enumerate(avg_growth.head(10).items(), 1):
    print(f"    {i:2d}. {country:<30} {growth:+.1f}%")

🏆 TOP TOURISM EARNERS

1️⃣ ALL-TIME TOP 10 TOURISM EARNERS:
     1. United States                  $3,929,432M
     2. France                         $1,319,503M
     3. Germany                        $1,047,240M
     4. Italy                          $689,676M
     5. Thailand                       $651,912M
     6. Australia                      $651,839M
     7. Hong Kong                      $529,183M
     8. Turkey                         $466,657M
     9. Macao                          $463,288M
    10. Japan                          $422,954M

2️⃣ TOP 10 IN 2020 (COVID year):
     1. United States                  $84,205M
     2. France                         $35,958M
     3. Australia                      $26,234M
     4. United Arab Emirates           $24,615M
     5. Italy                          $20,459M
     6. Austria                        $15,362M
     7. Thailand                       $15,360M
     8. Qatar                          $14,318M
     9. Turkey            